# Phase 2 — Synthetic Data Validation
## NeuroFlux Interactive Analysis Notebook

Runs the Phase 2 parameter sweep and produces all publication-quality figures
and statistical summaries described in `PLAN.md`.

**Quick start:** Run all cells top-to-bottom. By default this uses `quick_validation.yaml`
(2 runs, completes in seconds). Change `CONFIG = "full_sweep"` in the configuration
cell to run the full 5,760-experiment sweep (~15 min on 8 cores).

| Section | What it does |
|---|---|
| 1 | Run the parameter sweep → SQLite DB |
| 2 | Inspect aggregated results as a DataFrame |
| 3 | Generate SNR heatmap, convergence curves, STDP τ sensitivity |
| 4 | Statistical significance (t-test + Cohen's d) |
| 5 | Failure mode analysis |
| 6 | Export CSVs for paper / supplementary |

In [ ]:
import argparse
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from neuroflux_datasets.analysis import (
    compute_statistical_significance,
    fetch_aggregated_data,
    fetch_raw_runs,
    identify_failure_modes,
    plot_convergence_curves,
    plot_parameter_sensitivity,
    plot_snr_heatmap,
    summarize_significance,
)
from neuroflux_datasets.cli import run_experiment_command

# ── Configuration ────────────────────────────────────────────────────────────
# Change CONFIG to "full_sweep" to run the 5,760-experiment sweep.
CONFIG = "quick_validation"

# Project root is one level above experiments/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "phase2" / f"{CONFIG}.yaml"
DB_PATH = PROJECT_ROOT / "experiments" / f"{CONFIG}_results.db"
FIGURES_DIR = PROJECT_ROOT / "experiments" / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config  : {CONFIG_PATH}")
print(f"Database: {DB_PATH}")
print(f"Figures : {FIGURES_DIR}")

## Section 1 — Run the Parameter Sweep

Executes every combination in the config YAML and writes one row per run to SQLite.
If the database file already exists the sweep is skipped — delete the file to force
a re-run.

In [ ]:
if DB_PATH.exists():
    print("Database already exists — skipping sweep.")
    print(f"  {DB_PATH}")
    print("Delete the file to force a re-run.")
else:
    print(f"Running sweep: {CONFIG_PATH.name} ...")
    args = argparse.Namespace(config=str(CONFIG_PATH), output_db=str(DB_PATH))
    run_experiment_command(args)
    print("Sweep complete.")

conn = sqlite3.connect(str(DB_PATH))
row_count = conn.execute("SELECT COUNT(*) FROM runs").fetchone()[0]
conn.close()
print(f"\nTotal rows in database: {row_count}")

## Section 2 — Aggregated Results

Average ΔSNR over random seeds, grouped by
`(signal_type, flicker_freq, input_snr, epochs, stdp_tau)`.
Green = improvement; red = degradation.

In [ ]:
results = fetch_aggregated_data(DB_PATH)
df_agg = (
    pd.DataFrame(results)
    .sort_values(["signal_type", "input_snr", "flicker_freq"])
    .reset_index(drop=True)
)

print(f"Unique configurations: {len(df_agg)}")
print(f"Signal types present:  {sorted(df_agg['signal_type'].unique())}")
print()
display(
    df_agg.style.background_gradient(
        subset=["mean_delta_snr"], cmap="RdYlGn", vmin=-5, vmax=5
    ).format({"mean_delta_snr": "{:.3f}", "mean_convergence": "{:.1f}"})
)

## Section 3 — Publication Figures

Three figure types per signal class, saved as PNG to `experiments/figures/`:

- **SNR heatmap** — mean ΔSNR across Input SNR × Flicker Frequency
- **Convergence curves** — mean ΔSNR vs. training epochs, one line per input SNR
- **STDP τ sensitivity** — mean ΔSNR vs. STDP time constant

In [ ]:
signal_types = sorted(df_agg["signal_type"].unique())

for sig in signal_types:
    heatmap_path = FIGURES_DIR / f"snr_heatmap_{sig}.png"
    convergence_path = FIGURES_DIR / f"convergence_{sig}.png"
    sensitivity_path = FIGURES_DIR / f"sensitivity_{sig}.png"

    plot_snr_heatmap(results, sig, heatmap_path)
    plot_convergence_curves(results, sig, convergence_path)
    plot_parameter_sensitivity(results, sig, sensitivity_path)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, path, title in zip(
        axes,
        [heatmap_path, convergence_path, sensitivity_path],
        ["SNR Heatmap", "Convergence Curves", "STDP τ Sensitivity"],
        strict=False,
    ):
        ax.imshow(plt.imread(str(path)))
        ax.axis("off")
        ax.set_title(f"{sig.upper()} — {title}")

    fig.suptitle(f"Signal type: {sig.upper()}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(
        f"  Saved: {heatmap_path.name} | "
        f"{convergence_path.name} | {sensitivity_path.name}"
    )

## Section 4 — Statistical Significance

One-sample t-test per configuration: **H₀ = mean ΔSNR = 0** across seeds.
Effect size reported as Cohen's d.

**Acceptance criterion (PLAN.md):** ≥ 80% of configurations must show statistically
significant improvement (p < 0.01, paired t-test).

In [ ]:
raw_runs = fetch_raw_runs(DB_PATH)
sig_results = compute_statistical_significance(raw_runs, p_threshold=0.01)
df_sig = (
    pd.DataFrame(sig_results)
    .sort_values(["signal_type", "mean_delta_snr"], ascending=[True, False])
    .reset_index(drop=True)
)

display_cols = [
    "signal_type",
    "flicker_freq",
    "input_snr",
    "epochs",
    "stdp_tau",
    "n_seeds",
    "mean_delta_snr",
    "std_delta_snr",
    "t_stat",
    "p_value",
    "cohens_d",
    "significant",
    "improved",
]
display(
    df_sig[display_cols]
    .style.background_gradient(
        subset=["mean_delta_snr"], cmap="RdYlGn", vmin=-5, vmax=5
    )
    .format(
        {
            "mean_delta_snr": "{:.3f}",
            "std_delta_snr": "{:.3f}",
            "t_stat": "{:.2f}",
            "p_value": "{:.4f}",
            "cohens_d": "{:.2f}",
        },
        na_rep="—",
    )
)

In [ ]:
summary = summarize_significance(sig_results)

target_pct = 80.0
passed = summary["pct_significant"] >= target_pct
status = "✅ PASS" if passed else "❌ FAIL"

print("=" * 54)
print("SIGNIFICANCE SUMMARY")
print("=" * 54)
print(f"Total configurations      : {summary['total_configs']}")
n_imp, pct_imp = summary["improved_configs"], summary["pct_improved"]
print(f"Improved  (ΔSNR > 0)        : {n_imp:3d}  ({pct_imp:.1f}%)")
n_sig_c, pct_sig = summary["significant_configs"], summary["pct_significant"]
print(f"Significant (p < 0.01)    : {n_sig_c:3d}  ({pct_sig:.1f}%)")
print(f"Mean ΔSNR overall          : {summary['mean_delta_snr_overall']:.3f} dB")
print()
print(f"Acceptance criterion (≥{target_pct:.0f}% significant): {status}")

if summary["best_config"]:
    b = summary["best_config"]
    print(
        f"\nBest  config: {b['signal_type']:4s} | {b['flicker_freq']:5.0f} Hz | "
        f"{b['input_snr']:4.0f} dB | {b['epochs']:3d} ep | τ={b['stdp_tau']:.0f} ms "
        f"→ ΔSNR={b['mean_delta_snr']:.3f} dB"
    )

if summary["worst_config"]:
    w = summary["worst_config"]
    print(
        f"Worst config: {w['signal_type']:4s} | {w['flicker_freq']:5.0f} Hz | "
        f"{w['input_snr']:4.0f} dB | {w['epochs']:3d} ep | τ={w['stdp_tau']:.0f} ms "
        f"→ ΔSNR={w['mean_delta_snr']:.3f} dB"
    )

In [ ]:
print("Breakdown by signal type (p < 0.01):")
print(f"{'Signal':8s} {'Sig/Total':12s} {'%':6s} {'Mean ΔSNR':>12s}")
print("-" * 42)
for sig in signal_types:
    sub = df_sig[df_sig["signal_type"] == sig]
    n_sig = int(sub["significant"].sum())
    n_total = len(sub)
    pct = 100.0 * n_sig / n_total if n_total else 0.0
    mean_d = sub["mean_delta_snr"].mean()
    print(f"{sig:8s} {n_sig:3d}/{n_total:<7d}  {pct:5.1f}%  {mean_d:+.3f} dB")

## Section 5 — Failure Mode Analysis

Configurations where NeuroFlux underperformed. Three non-exclusive failure modes:

| Mode | Definition |
|---|---|
| `negative_improvement` | ΔSNR < 0 — SNN degraded the signal |
| `not_significant` | ΔSNR > 0 but p ≥ 0.01 — improvement not reliably reproducible |
| `high_variance` | σ(ΔSNR) > |mean ΔSNR| — high seed-to-seed variability |

In [ ]:
failures = identify_failure_modes(sig_results)

if failures:
    df_fail = (
        pd.DataFrame(failures).sort_values("mean_delta_snr").reset_index(drop=True)
    )
    print(f"Failure modes: {len(df_fail)} / {len(sig_results)} configurations")
    print()

    # Reason frequency
    from collections import Counter

    reason_counts: Counter = Counter()
    for r in failures:
        for reason in r["failure_reasons"].split(";"):
            reason_counts[reason] += 1
    for reason, count in sorted(reason_counts.items(), key=lambda x: -x[1]):
        print(f"  {reason:25s}: {count} configs")
    print()

    display(
        df_fail.style.background_gradient(
            subset=["mean_delta_snr"], cmap="Reds_r", vmin=-5, vmax=0
        ).format(
            {"mean_delta_snr": "{:.3f}", "std_delta_snr": "{:.3f}", "p_value": "{:.4f}"}
        )
    )
else:
    print("No failure modes identified across all configurations.")

## Section 6 — Export Results

Save full significance table and failure modes as CSV for paper / supplementary material.

In [ ]:
sig_csv = FIGURES_DIR / "significance_results.csv"
fail_csv = FIGURES_DIR / "failure_modes.csv"

df_sig.to_csv(sig_csv, index=False)
print(f"Saved: {sig_csv}")

if failures:
    df_fail.to_csv(fail_csv, index=False)
    print(f"Saved: {fail_csv}")
else:
    print("No failure modes — failure_modes.csv not written.")

print()
print("Figures directory:")
for p in sorted(FIGURES_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45s} {size_kb:6.1f} KB")

---
## Appendix — Running the Full Sweep

Change `CONFIG = "full_sweep"` in the configuration cell at the top and re-run.
The `ProcessPoolExecutor` in `run_experiment_command` scales to all available CPU cores.

| Machine | Cores | Estimated time |
|---------|-------|----------------|
| Laptop (4 cores) | 4 | ~30 min |
| Workstation (8 cores) | 8 | ~15 min |
| Server (16+ cores) | 16 | ~8 min |

Full sweep produces 5,760 rows (576 configs × 10 seeds).

CLI equivalent:
```bash
uv run neuroflux-experiment run configs/phase2/full_sweep.yaml --output-db results/full_sweep.db
uv run neuroflux-experiment analyze-significance --db results/full_sweep.db --output-dir results/
```